In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, r2_score
from sklearn.linear_model import LinearRegression
import os

## Load Datasets

In [2]:
base_path = os.pardir + "" + "\data\loan_default_datasets"

demographics = pd.read_csv(os.path.join(base_path, "traindemographics.csv"))
prevloans = pd.read_csv(os.path.join(base_path, "trainprevloans.csv"))
performance = pd.read_csv(os.path.join(base_path, "trainperf.csv"))


## Data Architecture & Merging

In [3]:
demographics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4346 entries, 0 to 4345
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customerid                  4346 non-null   object 
 1   birthdate                   4346 non-null   object 
 2   bank_account_type           4346 non-null   object 
 3   longitude_gps               4346 non-null   float64
 4   latitude_gps                4346 non-null   float64
 5   bank_name_clients           4346 non-null   object 
 6   bank_branch_clients         51 non-null     object 
 7   employment_status_clients   3698 non-null   object 
 8   level_of_education_clients  587 non-null    object 
dtypes: float64(2), object(7)
memory usage: 305.7+ KB


In [4]:
performance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4368 entries, 0 to 4367
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   customerid     4368 non-null   object 
 1   systemloanid   4368 non-null   int64  
 2   loannumber     4368 non-null   int64  
 3   approveddate   4368 non-null   object 
 4   creationdate   4368 non-null   object 
 5   loanamount     4368 non-null   float64
 6   totaldue       4368 non-null   float64
 7   termdays       4368 non-null   int64  
 8   referredby     587 non-null    object 
 9   good_bad_flag  4368 non-null   object 
dtypes: float64(2), int64(3), object(5)
memory usage: 341.4+ KB


In [5]:
prevloans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18183 entries, 0 to 18182
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customerid       18183 non-null  object 
 1   systemloanid     18183 non-null  int64  
 2   loannumber       18183 non-null  int64  
 3   approveddate     18183 non-null  object 
 4   creationdate     18183 non-null  object 
 5   loanamount       18183 non-null  float64
 6   totaldue         18183 non-null  float64
 7   termdays         18183 non-null  int64  
 8   closeddate       18183 non-null  object 
 9   referredby       1026 non-null   object 
 10  firstduedate     18183 non-null  object 
 11  firstrepaiddate  18183 non-null  object 
dtypes: float64(2), int64(3), object(7)
memory usage: 1.7+ MB


In [6]:
prevloans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18183 entries, 0 to 18182
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customerid       18183 non-null  object 
 1   systemloanid     18183 non-null  int64  
 2   loannumber       18183 non-null  int64  
 3   approveddate     18183 non-null  object 
 4   creationdate     18183 non-null  object 
 5   loanamount       18183 non-null  float64
 6   totaldue         18183 non-null  float64
 7   termdays         18183 non-null  int64  
 8   closeddate       18183 non-null  object 
 9   referredby       1026 non-null   object 
 10  firstduedate     18183 non-null  object 
 11  firstrepaiddate  18183 non-null  object 
dtypes: float64(2), int64(3), object(7)
memory usage: 1.7+ MB


In [12]:
prevloans_agg = prevloans.copy()

In [13]:
datetime_Columns = ['firstduedate', 'firstrepaiddate', 'creationdate', 'approveddate']

def convert_to_date(cols: list, df):
    for key in datetime_Columns:
        df[key] = pd.to_datetime(df[key], errors='coerce')

convert_to_date(datetime_Columns, prevloans)


In [10]:
# result = prevloans_agg.groupby('customerid').apply(lambda x: x['firstrepaiddate'] - x['firstduedate'])

In [14]:
prevloans_agg['days_late'] = (prevloans_agg['firstrepaiddate'] - prevloans_agg['firstduedate']).dt.days

In [15]:
prevloans_agg['expected_close_date'] = (prevloans_agg['approveddate'] + pd.to_timedelta(prevloans_agg['termdays'], 'D')).dt.date

In [16]:
prevloans_agg['expected_close_date'].dtype
prevloans_agg['closeddate'].dtype

dtype('O')

In [ ]:
datetime_Columns = ['firstduedate', 'firstrepaiddate', 'creationdate', 'approveddate', 'closeddate']

def convert_to_date(cols: list, df):
    for key in datetime_Columns:
        df[key] = pd.to_datetime(df[key], errors='coerce')

convert_to_date(datetime_Columns, prevloans)


In [18]:
prevloans_agg['total_settlement_delay'] = prevloans_agg['closeddate'] - prevloans_agg['expected_close_date']

TypeError: unsupported operand type(s) for -: 'str' and 'datetime.date'

In [ ]:
print(result[:5])

In [ ]:
type(result)

In [ ]:
prevloans_agg.columns

In [ ]:
prevloans_agg = prevloans_agg.groupby('customerid')

In [ ]:
prevloans_agg = prevloans_agg.agg(
    num_prev_loans=("systemloanid", 'count'),
    total_loan_amount=("loanamount", 'sum'),
    avg_loan_size=('loanamount', 'median'),
    first_installment_delay=("")
    )

In [ ]:
prevloans_agg.head()